
# Módulo 3 — 2/3 - Servicios de terceros: correo, redes y seguridad

En esta clase vamos a cubrir:

1. Por qué integrar scripts con servicios de terceros
2. Panorama: correo, almacenamiento en la nube y mensajería
3. Automatizar el correo con `smtplib` e `imaplib`
4. Automatizar redes sociales con Tweepy, Instabot y Facebook-sdk
5. Seguridad en las integraciones: claves, SQL y mensajes de error
6. Un quiz interactivo para repasar

> Idea central de la clase: **tus scripts son islas y las integraciones son puentes**. La
> automatización real aparece cuando un script puede hablar con un servicio que ya resuelve
> un problema difícil: mandar correo, publicar en una red social, guardar en la nube.

Comprobamos el entorno antes de empezar.

In [1]:
import sys
from datetime import datetime

print("Python:", sys.version.split()[0])
print("Ahora:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Python: 3.13.3
Ahora: 2026-10-26 19:11:09


---

## 1. Por qué integrar con servicios de terceros

Un script suelto vive en tu máquina y hace lo que le pediste. Integrarlo con un servicio
de terceros significa que **ese servicio hace el trabajo pesado** y tu script solo coordina.

Los tres escenarios del módulo:

- **Correo**: enviar confirmaciones, informes semanales, alertas cuando llega un correo con
  cierta palabra clave.
- **Nube**: copias de seguridad automáticas, sincronización de archivos, nada de documentos
  perdidos.
- **Mensajería**: chatbots que responden consultas, envían novedades y resumen conversaciones
  para que el chat no se vaya de las manos.

Los cuatro beneficios que da una integración:

| Beneficio | Qué significa | Ejemplo |
|-----------|---------------|---------|
| **Funcionalidad ampliada** | No reinventar lo que ya existe | Usar una pasarela de pago en vez de construir un sistema de pagos |
| **Eficiencia** | El trabajo repetitivo lo hace la máquina | Hojas de cálculo → CRM con scripts de entrada y seguimiento |
| **Poder de los datos** | Analítica sin construirla | Tráfico web, comportamiento de usuario, ROI de campañas |
| **Escalabilidad** | Crecer sin comprar hardware | OneDrive cuando la base de usuarios se dispara |

Y una advertencia honesta: automatizar **no reemplaza** a la persona, la potencia. Y cuanto
más dependas de servicios externos, más importance tienen las copias de seguridad propias.

In [2]:
# Los 4 beneficios como dato, para poder iterarlos o documentarlos
BENEFICIOS = {
    "Funcionalidad ampliada": "No reinventar la rueda: usar un servicio que ya resuelve el problema.",
    "Eficiencia":             "El trabajo repetitivo lo hace la maquina, no la persona.",
    "Poder de los datos":     "Analitica y metricas sin construir la infraestructura.",
    "Escalabilidad":          "Crecer sin comprar ni mantener hardware propio.",
}

for nombre, descripcion in BENEFICIOS.items():
    print(f"\n{nombre}")
    print(f"  {descripcion}")


Funcionalidad ampliada
  No reinventar la rueda: usar un servicio que ya resuelve el problema.

Eficiencia
  El trabajo repetitivo lo hace la maquina, no la persona.

Poder de los datos
  Analitica y metricas sin construir la infraestructura.

Escalabilidad
  Crecer sin comprar ni mantener hardware propio.


---

## 2. Panorama: qué servicio usar para qué

Antes de escribir código, elegí el servicio. Esta tabla es la que te va a ahorrar
escribir algo que después tiras:

| Necesitás | Servicio | Librería de Python | Caso típico |
|-----------|----------|--------------------|-------------|
| Enviar correo | SMTP | `smtplib` + `email.mime.text.MIMEText` | Informes, confirmaciones, alertas |
| Leer la bandeja | IMAP | `imaplib` | Filtrar no leídos, buscar por remitente |
| Publicar en Slack / Teams | API de Slack | `slack_sdk` | Chatbot que responde preguntas |
| Publicar en Instagram | API de Instagram | `instabot` | Me gusta, comentarios, seguir |
| Publicar en X / Twitter | API de X | `tweepy` | Publicar, menciones, sentimiento |
| Gestionar páginas | API de Facebook | `facebook-sdk` | Publicaciones de página |
| Pagos | Pasarela (Stripe, PayPal) | SDK del proveedor | Checkout sin PCI propio |
| Guardar archivos | Almacenamiento (OneDrive, S3) | SDK del proveedor | Backups y sincronización |

Todas estas librerías siguen el mismo patrón del módulo anterior: **el token va en una
variable de entorno** y los errores se capturan con `try/except`.

### Ejemplo con Slack

Este es el chatbot del módulo de referencia. `chat_postMessage()` publica, y `SlackApiError`
captura el caso de que el token caducó o el canal no existe.

> No lo ejecutamos: necesita el `slack_sdk` instalado y un token real del bot. La estructura
> es exactamente la que usarías en producción.

In [3]:
# --- Ejemplo ilustrativo (no se ejecuta) ---
# from slack_sdk import WebClient
# from slack_sdk.errors import SlackApiError
#
# slack_token = "xoxb-..."          # <- token del bot, NUNCA en el codigo
# client = WebClient(token=slack_token)
# channel_id = "C0123456789"
#
# def handle_message(event):
#     text = event["text"].lower()
#     if "hello" in text:
#         response = "Hi there! How can I help you today?"
#     elif "weather" in text:
#         response = "The weather in Wyoming, MI is currently sunny and 72F."
#     else:
#         response = "I'm not sure I understand. Try asking me about the weather."
#     try:
#         client.chat_postMessage(channel=channel_id, text=response)
#     except SlackApiError as e:
#         print(f"Error posting message: {e}")

print("El patron es siempre el mismo: token -> cliente -> try/except alrededor de la llamada.")

El patron es siempre el mismo: token -> cliente -> try/except alrededor de la llamada.


In [4]:
# Podemos simular la logica del bot sin necesidad de Slack
RESPUESTAS = {
    "hello":   "Hi there! How can I help you today?",
    "weather": "The weather in Wyoming, MI is currently sunny and 72F.",
}

def responder(mensaje):
    clave = mensaje.lower()
    if clave in RESPUESTAS:
        return RESPUESTAS[clave]
    return "I'm not sure I understand. Try asking me about the weather."

for mensaje in ("Hello", "weather", "quiero un cafe"):
    print(f"{mensaje:16} -> {responder(mensaje)}")

Hello            -> Hi there! How can I help you today?
weather          -> The weather in Wyoming, MI is currently sunny and 72F.
quiero un cafe   -> I'm not sure I understand. Try asking me about the weather.


---

## 3. Automatizar el correo: smtplib e imaplib

El correo tiene dos protocolos y una cosa parece email:

- **SMTP** (`smtplib`) = **enviar**
- **IMAP** (`imaplib`) = **leer** la bandeja, buscar, marcar, mover y borrar

Para construir el mensaje se usa `email.mime.text.MIMEText`, que crea un **objeto de correo**
en vez de una cadena suelta. La diferencia importa: un objeto sabe qué es cada cabecera.

Usos reales que aparecen en el módulo:

- Informe de ventas semanal automático (con el gráfico hecho en `matplotlib`)
- Atención al cliente: correos urgentes por palabra clave → acuse de recibo → derivación
- Notificaciones de redes sociales por correo
- Backups y archivado de la bandeja

### Construir el mensaje

`MIMEText` es un objeto. Podemos inspeccionarlo antes de mandarlo, que es la mejor manera de
depurar: si el aviso está mal, el problema casi siempre está en las cabeceras.

In [5]:
from email.mime.text import MIMEText

remitente = "notificaciones@ejemplo.com"
destinatario = "ana@ejemplo.com"

mensaje = MIMEText(
    "Hola Ana,\n\nTu pedido B-204 ha sido enviado.\n\nSaludos,\nEl equipo",
)
mensaje["Subject"] = "Confirmacion de pedido B-204"
mensaje["From"] = remitente
mensaje["To"] = destinatario

# as_string() devuelve el correo completo tal y como se enviara
print(mensaje.as_string())

Content-Type: text/plain; charset="us-ascii"
MIME-Version: 1.0
Content-Transfer-Encoding: 7bit
Subject: Confirmacion de pedido B-204
From: notificaciones@ejemplo.com
To: ana@ejemplo.com

Hola Ana,

Tu pedido B-204 ha sido enviado.

Saludos,
El equipo


Fijate en lo que devuelve `as_string()`: `Content-Type`, `MIME-Version`, `Date`, y las
cabeceras `Subject`, `From` y `To` en su forma final. Si algo va mal al enviar, es
exactamente este texto el que hay que mirar primero.

In [6]:
# Los adjuntos y el HTML son el siguiente escalon
from email.mime.multipart import MIMEMultipart
from email.mime.application import MIMEApplication

def construir_correo_con_adjunto(asunto, cuerpo, ruta_adjunto):
    # MIMEMultipart hace de "sobre" y permite mezclar texto, HTML y adjuntos
    correo = MIMEMultipart()
    correo["Subject"] = asunto
    correo["From"] = remitente
    correo["To"] = destinatario

    correo.attach(MIMEText(cuerpo, "plain"))

    with open(ruta_adjunto, "rb") as archivo:
        adjunto = MIMEApplication(archivo.read())
        adjunto.add_header(
            "Content-Disposition", "attachment", filename=ruta_adjunto.split("/")[-1]
        )
        correo.attach(adjunto)

    return correo

# Construimos uno de prueba en memoria para ver la estructura
correo = MIMEMultipart()
correo["Subject"] = "Informe semanal"
correo["From"] = remitente
correo["To"] = destinatario
correo.attach(MIMEText("Adjunto el informe en PDF.", "plain"))
adjunto = MIMEApplication(b"%PDF-1.4 datos de prueba", Name="informe.pdf")
adjunto["Content-Disposition"] = "attachment; filename=informe.pdf"
correo.attach(adjunto)

print(correo.as_string()[:420], "...")

Content-Type: multipart/mixed; boundary="===============0424550314681525270=="
MIME-Version: 1.0
Subject: Informe semanal
From: notificaciones@ejemplo.com
To: ana@ejemplo.com

--===============0424550314681525270==
Content-Type: text/plain; charset="us-ascii"
MIME-Version: 1.0
Content-Transfer-Encoding: 7bit

Adjunto el informe en PDF.
--===============0424550314681525270==
Content-Type: application/octet-stream; Nam ...


### Leer la bandeja con IMAP

La secuencia de `imaplib` es siempre la misma:

| Paso | Método | Qué hace |
|------|--------|----------|
| 1 | `IMAP4_SSL(servidor, puerto)` | Conecta cifrado (puerto 993) |
| 2 | `login(usuario, contrasena)` | Autentica |
| 3 | `select("INBOX")` | Elige el buzón |
| 4 | `search(None, "UNSEEN")` | Busca: no leídos, por remitente, por fecha... |
| 5 | `fetch(num, "(RFC822)")` | Descarga el correo crudo |
| 6 | `close()` / `logout()` | Cierra y se despide |

Fíjate en `search(None, "UNSEEN FROM \"user@example.com\"")`: los criterios se encadenan.

In [ ]:
# --- Estructura real de conexion IMAP (NO se ejecuta: falta servidor y contrasena) ---
# import imaplib
#
# imap_server = "imap.ejemplo.com"
# imap_port = 993
# email_address = "cuenta@ejemplo.com"
# email_password = os.environ["EMAIL_PASSWORD"]     # <- del entorno, nunca escrita
#
# mail = imaplib.IMAP4_SSL(imap_server, imap_port)
# mail.login(email_address, email_password)
# mail.select("INBOX")
#
# status, messages = mail.search(None, "UNSEEN")
# for num in messages[0].split():
#     status, data = mail.fetch(num, "(RFC822)")
#     # data[0][1] contiene el correo en crudo
#
# mail.close()
# mail.logout()

print("IMAP: conectar -> autenticar -> seleccionar buzon -> buscar -> descargar -> cerrar")

Lo interesante es lo que se hace **con** el correo una vez descargado. `fetch()` devuelve
**bytes**, no un objeto: hay que convertirlo con `email.message_from_bytes()`.

Podemos practicarlo con un correo simulado, sin necesidad de servidor.

In [7]:
import email
from email import policy

CORREO_SIMULADO = """From: Ana Lopez <ana@ejemplo.com>
To: soporte@ejemplo.com
Subject: URGENTE: no puedo acceder a mi cuenta
Date: Tue, 13 Oct 2026 09:41:22 -0300
Content-Type: text/plain; charset="utf-8"

Hola, desde ayer no puedo entrar a mi cuenta.
Es urgente, tengo una reunion manana.
"""

# Los bytes son lo que devuelve fetch() de IMAP
crudo = CORREO_SIMULADO.encode("utf-8")
mensaje = email.message_from_bytes(crudo, policy=policy.default)

print("De:      ", mensaje["From"])
print("Asunto:  ", mensaje["Subject"])
print("Fecha:   ", mensaje["Date"])
print()
print("Cuerpo:")
print(mensaje.get_content().strip())

De:       Ana Lopez <ana@ejemplo.com>
Asunto:   URGENTE: no puedo acceder a mi cuenta
Fecha:    Tue, 13 Oct 2026 09:41:22 -0300

Cuerpo:
Hola, desde ayer no puedo entrar a mi cuenta.
Es urgente, tengo una reunion manana.


In [ ]:
# El paso siguiente es DECIDIR QUE HACER con cada correo: eso es automatizar de verdad
PALABRAS_URGENTES = ("urgente", "urgencia", "no puedo", "bloqueado")
CLIENTES_VIP = ("ana@ejemplo.com", "mora@ejemplo.com")

def clasificar(mensaje):
    remitente = mensaje["From"]
    cuerpo = mensaje.get_content().lower()

    if remitente in CLIENTES_VIP or "urgente" in cuerpo:
        return "PRIORIDAD ALTA", "derivar a un agente humano"
    if any(palabra in cuerpo for palabra in PALABRAS_URGENTES):
        return "REVISAR", "responder con acuse de recibo"
    return "NORMAL", "archivar"

ASUNTOS = (
    "URGENTE: no puedo acceder a mi cuenta",
    "Consulta sobre la factura del mes",
    "Reunion el jueves a las 10",
)

for asunto in ASUNTOS:
    simulado = (
        "From: cliente@ejemplo.com\n"
        "To: soporte@ejemplo.com\n"
        f"Subject: {asunto}\n"
        "\n"
        "Hola, escribo para consultar lo siguiente.\n"
    )
    prioridad, accion = clasificar(email.message_from_string(simulado, policy=policy.default))
    print(f"{asunto:42} -> {prioridad:16} {accion}")

### Enviar con SMTP, con las buenas prácticas

Las tres que el módulo repite:

1. **Nunca hardcodear la contraseña**: variable de entorno o configuración cifrada
2. **`time.sleep()` entre envíos**: los proveedores de correo tienen límites y si los
   superás te marcan como spam
3. **`try/except` siempre**: un servidor que cae a mitad del bucle no puede dejar el
   script colgado sin avisar

Y un consejo general: **probá con un servidor de pruebas** antes de tocar producción. SendGrid
ofrece un SMTP gratis para experimenting.

In [8]:
# --- Envio real (NO se ejecuta: necesita servidor y contrasena) ---
# import smtplib, ssl
#
# smtp_server = "smtp.ejemplo.com"
# smtp_port = 587
# USUARIO = os.environ["EMAIL_USER"]
# CLAVE = os.environ["EMAIL_PASSWORD"]      # <- del entorno
#
# contexto = ssl.create_default_context()
# with smtplib.SMTP(smtp_server, smtp_port) as servidor:
#     servidor.starttls(context=contexto)
#     servidor.login(USUARIO, CLAVE)
#     servidor.send_message(mensaje)
#     print("Correo enviado")

# Lo que SI podemos ejecutar: el envio en seco, que solo imprime lo que se mandaria
def enviar_en_seco(asunto, cuerpo, destinatario):
    correo = MIMEText(cuerpo)
    correo["Subject"] = asunto
    correo["From"] = "notificaciones@ejemplo.com"
    correo["To"] = destinatario
    print("--- se enviaria este correo ---")
    print(correo.as_string())

enviar_en_seco(
    "Confirmacion de pedido B-204",
    "Hola Ana,\n\nTu pedido B-204 ha sido enviado.\n\nSaludos,\nEl equipo",
    "ana@ejemplo.com",
)

--- se enviaria este correo ---
Content-Type: text/plain; charset="us-ascii"
MIME-Version: 1.0
Content-Transfer-Encoding: 7bit
Subject: Confirmacion de pedido B-204
From: notificaciones@ejemplo.com
To: ana@ejemplo.com

Hola Ana,

Tu pedido B-204 ha sido enviado.

Saludos,
El equipo


### El reto del módulo

Unificar las dos mitades: **mandar** una confirmación y **comprobar** si el cliente responde.

La estructura que se pide es esta, con la lógica de negocio separada del envío:

```python
def send_confirmation_email(client_email, client_name):
    # construir el MIMEText y enviarlo con smtplib
    ...

def check_new_messages(client_email, client_name):
    # conectarse con IMAP4_SSL, buscar no leidos, ver si hay respuesta
    ...
```

Lo que hay que respetar:

- Servidor SMTP `smtp.ejemplo.com:587` e IMAP `imap.ejemplo.com:993`
- La contraseña **nunca** en el archivo
- `time.sleep()` entre envíos para no parecer spam

---

## 4. Automatizar redes sociales

Cada red tiene su propio SDK. Son librerías de bloqueo: te ahorran el trabajo de
autenticarte y de conocer la API de cada plataforma.

| Librería | Plataforma | Qué permite |
|----------|-----------|-------------|
| `tweepy` | X / Twitter | Publicar, retuitear, seguir, buscar menciones |
| `instabot` | Instagram | Me gusta, seguir, comentar, responder |
| `facebook-sdk` | Facebook | Publicar y administrar páginas |

Los casos del módulo:

- **Publicación programada**: una blogger programa una semana de contenido de golpe y lo
  publica en las horas de más interacción
- **Recuperación de datos**: hashtags en tendencia, menciones de marca para vigilar la
  reputación, análisis de sentimiento sobre el último producto
- **Interacción**: responder comentarios, dar me gusta, seguir

Y tres dudas que se repiten en el material:
1. Cada plataforma tiene sus normas: incumplirlas es suspensión de cuenta
2. Contenido genérico = spam: la automatización no puede sustituir la voz de la persona
3. Es una herramienta, no un sustituto de la interacción humana real

### El reto: publicar en dos horarios

El ejercicio pide programar una misma función a las 10:00 y a las 16:00. Resuelto con
`schedule`, el módulo que vemos en detalle en el notebook 3/3.

In [ ]:
import schedule
import time
from datetime import datetime

def promote_product():
    # Un solo cuerpo, publicado en las dos plataformas
    mensaje = "Alerta de nuevo producto! Mirá https://www.example.com/new-product #newproduct"
    print(f"[{datetime.now():%H:%M:%S}] Posting to Twitter: {mensaje}")
    print(f"[{datetime.now():%H:%M:%S}] POST en Facebook: Por fin esta aqui nuestro ultimo producto!")

# Programacion: todos los dias a las 10:00 y a las 16:00
schedule.every().day.at("10:00").do(promote_product)
schedule.every().day.at("16:00").do(promote_product)

# Inspeccionamos lo que quedo programado
for trabajo in schedule.get_jobs():
    print(f"  {str(trabajo.at_time):8} -> {trabajo.job_func.__name__}")
    print(f"  proxima ejecucion: {trabajo.next_run}")

In [ ]:
# El bucle real de un planificador es infinito, asi que aqui lo limitamos a 3 vueltas
# para poder ejecutarlo en el notebook sin colgarse.
def run_scheduler(vueltas=3):
    for _ in range(vueltas):
        schedule.run_pending()
        time.sleep(1)

# Para ejecutar esto en tu maquina de verdad, quitale el limite:
#     while True:
#         schedule.run_pending()
#         time.sleep(1)

print("Ejecutando el planificador 3 vueltas...")
run_scheduler(vueltas=3)
print("Terminado. Ahora las tareas programmingadas se ejecutan cuando corresponda.")

In [ ]:
# Y para limpiar las tareas si queres reutilizar el kernel sin reiniciarlo
print("Tareas antes:", len(schedule.get_jobs()))
schedule.clear()
print("Tareas despues:", len(schedule.get_jobs()))

---

## 5. Seguridad en las integraciones

La metáfora del módulo es que **el código es la fortaleza y las integraciones son las
puertas**: cada servicio al que llamás es un punto de entrada para un atacante.

Las cinco prácticas:

1. **Validar las entradas**: nunca confíes ciegamente en lo que llega del usuario o de la API
2. **Parametrizar las consultas**: contra inyección SQL
3. **No hardcodear claves**: variables de entorno o configuración cifrada
4. **Errores que no filtran información**: los mensajes de error son un mapa para el atacante
5. **Comunicaciones seguras**: HTTPS

Además: cifrado en tránsito y en reposo, retención mínima de datos, y auditorías
periódicas. La seguridad es un compromiso continuo, no un estado que se alcanza.

In [ ]:
import os

# --- Antipatron: la clave viaja en el codigo, y con ella en cada commit ---
CLAVE_MAL = "a1b2c3d4e5f6g7h8i9j0-mi-clave-real-de-produccion"

# --- Patron correcto: del entorno, con fallo elegante si falta ---
def obtener_clave(nombre):
    clave = os.environ.get(nombre)
    if not clave:
        raise SystemExit(
            f"Falta la variable de entorno {nombre}.\n"
            f"Definila con:  setx {nombre} \"tu-clave\"   (Windows)\n"
            f"No la escribas en el codigo ni la subas al repositorio."
        )
    return clave

try:
    clave = obtener_clave("SLACK_BOT_TOKEN")
except SystemExit as e:
    clave = None
    print(e)

print()
print("La clave hardcodeada que escribimos:", len(CLAVE_MAL), "caracteres en el archivo")
print("Desde el entorno:", "cargada" if clave else "no configurada (y el script lo dice)")

In [ ]:
# --- Inyeccion SQL: la diferencia entre concatenar y parametrizar ---
# Un nombre de usuario malicioso:  ana' OR '1'='1

# MAL: se construye el texto de la consulta con el dato del usuario
usuario_malicioso = "ana' OR '1'='1"
consulta_mala = f"SELECT * FROM usuarios WHERE nombre = '{usuario_malicioso}'"
print("Consulta MAL:")
print("  ", consulta_mala)
print("  -> El usuario logro cerrar la comilla y OR 1=1 devuelve TODOS los usuarios.\n")

# BIEN: el valor viaja como parametro, separado de la consulta
consulta_buena = "SELECT * FROM usuarios WHERE nombre = ?"
parametros = (usuario_malicioso,)
print("Consulta BIEN:")
print("  ", consulta_buena)
print("   parametros:", parametros)
print("  -> El motor trata la comilla como un caracter normal. No se inyecta nada.")

In [ ]:
# --- Errores: informativos para el usuario, ciegos para el atacante ---
class ErrorDeBaseDeDatos(Exception):
    pass

def consultar_usuario(nombre, conexion=None):
    if not nombre or not nombre.strip():
        raise ValueError("El nombre no puede estar vacio.")

    if conexion is None:
        # Error interno: registramos el detalle, pero al usuario le llega otra cosa
        detalle = "host=db-prod-01 user=app_admin password=hunter22 SELECT ... usuarios"
        print("[LOG INTERNO] Error de conexion:", detalle)
        raise ErrorDeBaseDeDatos("No pudimos consultar el usuario en este momento.")

    return {"nombre": nombre}

try:
    consultar_usuario("   ")
except ValueError as e:
    print("[ERROR DE ENTRADA]", e)

try:
    consultar_usuario("ana")
except ErrorDeBaseDeDatos as e:
    print("[ERROR PARA EL USUARIO]", e)
    print("  -> Ni la IP del servidor, ni el usuario de la BD, ni la consulta.")

### HTTPS

Todo lo que vimos en el notebook 1 y en esta clase debe ir por HTTPS. HTTP "seguro" es la
cifra al revés: la `S` de https es la de **cifrado**.

---

# Quiz interactivo: Servicios de terceros

Responde cada pregunta y pulsa **Ver respuesta**. Hay preguntas de una sola opcion y de multiple respuesta (selecciona todas las que correspondan). Si fallas, lee la explicacion: ahi esta el porque.

In [ ]:
# --- Configuracion del quiz ---
# 1) Localizar la raiz del proyecto (la que contiene la carpeta 'quiz') y anadirla al sys.path
import sys
from pathlib import Path

RAIZ = Path.cwd()
while RAIZ.parent != RAIZ and not (RAIZ / "quiz").exists():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.append(str(RAIZ))

# 2) Importar funciones del modulo reutilizable
from quiz.quiz import aplicar_estilos, crear_quiz, crear_quiz_multiple

# 3) Aplicar el CSS (legible en VS Code y JupyterLab)
aplicar_estilos()

# 4) Cargar respuestas y explicaciones desde el JSON (misma carpeta que el notebook)
import json
with open(Path.cwd() / "Modulo_3_Servicios_Terceros_preguntas.json", encoding="utf-8") as fh:
    respuestas = json.load(fh)

print("Quiz listo:", ", ".join(respuestas.keys()))

### Pregunta 1

Tu equipo tiene dificultades para seguir el ritmo del volumen de mensajes y notificaciones en su plataforma de chat. Quieres automatizar parte de la comunicación para mejorar la eficiencia. ¿Cuál de las siguientes soluciones podría implementarse utilizando Python y un servicio de terceros?

- Desarrolle un script que filtre los mensajes, envíe actualizaciones automáticas y resuma las conversaciones.
- Contratar a un becario para que gestione y responda manualmente a todos los mensajes y notificaciones.
- Cambie a otra plataforma de chat en equipo que tenga funciones de automatización integradas.
- Cree un sistema de archivo físico para organizar las copias impresas de todos los mensajes.

In [ ]:
q1_data = respuestas["q1"]
crear_quiz(q1_data["opciones"], q1_data["correcta"], q1_data["explicacion"])

### Pregunta 2

Su empresa experimenta cortes ocasionales del sistema que interrumpen el acceso de los empleados a los archivos de trabajo esenciales. ¿Qué solución de automatización, utilizando Python y un servicio de terceros, mitigaría mejor el impacto y garantizaría que los empleados tengan acceso a los archivos durante las interrupciones?

- Implemente copias de seguridad automatizadas de archivos críticos en un servicio de almacenamiento en la nube.
- Utiliza Python para crear un servidor local que replique todos los archivos almacenados en el servidor principal de la empresa.
- Automatice el proceso de envío de correos electrónicos a Asistencia de TI cuando se detecte una interrupción.
- Desarrolle un chatbot que notifique a los empleados cuando se produzca una interrupción.

In [ ]:
q2_data = respuestas["q2"]
crear_quiz(q2_data["opciones"], q2_data["correcta"], q2_data["explicacion"])

### Pregunta 3

El equipo de ventas usa hojas de cálculo para seguir a los clientes potenciales, con una gestión eficiente de los datos y alto riesgo de errores. ¿Qué solución, utilizando Python e integraciones de terceros, mejoraría el flujo de trabajo del equipo?

- Utilice Python para crear una aplicación web personalizada que reproduzca la funcionalidad de su actual sistema de hojas de cálculo.
- Desarrollar un script en Python que genere automáticamente informes de ventas a partir de los datos de la hoja de cálculo.
- Crear un script en Python que envíe recordatorios automáticos al equipo de ventas para que actualice la hoja de cálculo con regularidad.
- Integre su sistema de administración de relaciones con clientes (CRM) con scripts de Python para automatizar la introducción de datos, la asignación de clientes potenciales y los correos electrónicos de seguimiento.

In [ ]:
q3_data = respuestas["q3"]
crear_quiz(q3_data["opciones"], q3_data["correcta"], q3_data["explicacion"])

### Pregunta 4

Es responsable de gestionar una bandeja de entrada de atención al cliente con mucho volumen y necesita priorizar las consultas urgentes de clientes específicos. ¿Qué enfoque, utilizando Python y la biblioteca imaplib, sería más eficaz?

- Escriba un script que compruebe periódicamente si hay nuevos correos, identifique los que proceden de clientes específicos o contienen determinadas palabras clave y le envíe notificaciones.
- Utilice Python para reenviar automáticamente todos los mensajes de clientes específicos a una bandeja de entrada independiente de alta prioridad.
- Escribe un script que elimine automáticamente todos los correos electrónicos que no sean de clientes específicos.
- Desarrolle un script que responda automáticamente a cada correo electrónico con un mensaje genérico de acuse de recibo.

In [ ]:
q4_data = respuestas["q4"]
crear_quiz(q4_data["opciones"], q4_data["correcta"], q4_data["explicacion"])

### Pregunta 5

Emily, una popular artista, quiere automatizar las interacciones con sus seguidores en Instagram para aumentar la participación y construir una comunidad más fuerte. ¿Qué acciones podría automatizar utilizando Python y la biblioteca Instabot?

- Seguir cuentas que interactúen frecuentemente con su contenido.
- Me gusta de los POST de sus seguidores.
- Enviar respuestas personalizadas a los comentarios de sus POST.
- Crear automáticamente nuevas obras de arte y publicarlas en su feed.

In [ ]:
q5_data = respuestas["q5"]
crear_quiz_multiple(q5_data["opciones"], q5_data["correctas"], q5_data["explicacion"])

### Pregunta 6

Su equipo desarrolla una aplicación web que se integra con una pasarela de pago de terceros para procesar transacciones en línea. ¿Qué medidas de seguridad deben priorizarse para proteger los datos confidenciales de los clientes durante esta integración?

- Audite y actualice periódicamente la integración en busca de vulnerabilidades de seguridad.
- Implantar mecanismos sólidos de autenticación y autorización.
- Almacene las claves y contraseñas de la API directamente en el código fuente de la aplicación.
- Cifrado de datos en tránsito y en reposo.

In [ ]:
q6_data = respuestas["q6"]
crear_quiz_multiple(q6_data["opciones"], q6_data["correctas"], q6_data["explicacion"])

## ¡Fin del quiz!

Si fallaste alguna, releé la explicación. Fíjate especialmente en la pregunta 6: guardar
claves en el código fuente es el error que más caro sale en la vida real, y el mismo que
vimos en el notebook 1/3 con las variables de entorno.